# PLAN classification  training and validation

This notebook uses `data/processed_plan/panel.csv` produced by the B2-backed PLAN builder. It evaluates the baseline, logistic regression, and histogram gradient boosting models with and without trailing solar features.

**Test split isolation:** this notebook queries only `train` and `valid` rows. It performs grouped five-fold cross-validation on training objects, uses validation for model selection and threshold locking, and writes the frozen fitted pipelines for the separate test notebook. Never use test performance to change model or threshold choices.

## Data, feature guard, and split checks

The target is `reenters_within_horizon`; features are loaded from the built manifest. The explicit denylist supplements the manifest so the training lists cannot include target, outcome, or post-outcome fields.

In [ ]:
import json
from pathlib import Path
import duckdb
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits
SEED = 20261021
BOOTSTRAPS = 500
MAX_FALSE_ALARM_RATE = 0.10
TARGET = "reenters_within_horizon"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/processed_plan/panel.csv").is_file())
DATA_DIR = ROOT / "data/processed_plan"
PANEL_PATH = DATA_DIR / "panel.csv"
FEATURE_GROUPS = json.loads((DATA_DIR / "feature_groups.json").read_text(encoding="utf-8"))
NO_SUN = FEATURE_GROUPS["ablation"]["baseline_no_sun"]
WITH_SUN = FEATURE_GROUPS["ablation"]["with_sun_trailing"]
TRAILING_WEATHER = FEATURE_GROUPS["features"]["space_weather_trailing"]
NEVER_FEATURES = set(FEATURE_GROUPS.get("leakage_never_use_as_features", [])) | {
    "lifetime_days", "decay_date", "is_decayed", "days_followed", "days_remaining",
    "log1p_days_remaining", "reenters_within_horizon", "event_observed",
    "n_decay_msgs", "sw_asof_date", "decay_pred_spread_days", "object_name", "object_id",
}
for feature_set in [NO_SUN, WITH_SUN]:
    overlap = sorted(set(feature_set) & NEVER_FEATURES)
    assert not overlap, f"Forbidden feature(s): {overlap}"
    assert not any(column.startswith("fwd_") for column in feature_set), "Forward solar features are disabled by PLAN."
assert set(WITH_SUN) - set(NO_SUN) == set(TRAILING_WEATHER)
# DuckDB projects only needed columns and filters before materializing rows.
# The training/validation notebook never creates a test mask or test dataframe.
needed = sorted(set(NO_SUN + WITH_SUN + [
    "split", "norad_cat_id", "obs_date", TARGET, "f107_adj_mean_81d",
    "perigee_km_asof", "perigee_decay_rate_km_per_day", "object_type", "rcs_size",
]))
quoted = ", ".join('"' + column.replace('"', '""') + '"' for column in needed)
data = duckdb.connect().execute(
    f"SELECT {quoted} FROM read_csv_auto(?) WHERE split IN ('train', 'valid')",
    [str(PANEL_PATH)],
).df()
data["obs_date"] = pd.to_datetime(data["obs_date"])
assert set(data["split"].unique()) <= {"train", "valid"}
assert TARGET == FEATURE_GROUPS["targets"]["classification"]
train = data["split"].eq("train")
valid = data["split"].eq("valid")
assert train.any() and valid.any()
assert not set(data.loc[train, "norad_cat_id"]) & set(data.loc[valid, "norad_cat_id"])
assert set(data.loc[train, "obs_date"].dt.year).issubset(set(range(2015, 2021)))
assert data.loc[valid, "obs_date"].dt.year.eq(2022).all()
print("Loaded train and validation rows only; test rows were not materialized.")
for name, mask in [("Train", train), ("Validation", valid)]:
    subset = data.loc[mask]
    print(f"{name}: {len(subset):,} rows; {subset.norad_cat_id.nunique():,} objects; "
          f"positive rate {subset[TARGET].mean():.4f}")
def make_model(features, algorithm):
    numeric = data.loc[train, features].select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical = [column for column in features if column not in numeric]
    numeric_steps = [("imputer", SimpleImputer(strategy="median", add_indicator=True))]
    if algorithm == "logistic":
        numeric_steps.append(("scale", StandardScaler()))
    preprocess = ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=algorithm == "logistic")),
        ]), categorical),
    ], sparse_threshold=0.3 if algorithm == "logistic" else 0)
    if algorithm == "logistic":
        classifier = LogisticRegression(class_weight="balanced", max_iter=1000, solver="lbfgs", random_state=SEED)
    else:
        classifier = HistGradientBoostingClassifier(
            learning_rate=0.1, max_iter=100, max_leaf_nodes=31, min_samples_leaf=20,
            l2_regularization=0.0, class_weight="balanced", early_stopping=False, random_state=SEED,
        )
    return Pipeline([("preprocess", preprocess), ("classifier", classifier)])
def recall_at_fpr(y, probability, sample_weight=None):
    fpr, tpr, _ = roc_curve(y, probability, sample_weight=sample_weight)
    allowed = fpr <= MAX_FALSE_ALARM_RATE
    return float(tpr[allowed].max()) if allowed.any() else 0.0
def metric_values(y, probability, sample_weight=None):
    probability = np.asarray(probability, dtype=float)
    prediction = probability >= 0.5
    return {
        "Accuracy (threshold 0.50)": float(accuracy_score(y, prediction, sample_weight=sample_weight)),
        "Precision (threshold 0.50)": float(precision_score(y, prediction, sample_weight=sample_weight, zero_division=0)),
        "Recall (threshold 0.50)": float(recall_score(y, prediction, sample_weight=sample_weight, zero_division=0)),
        "ROC AUC": float(roc_auc_score(y, probability, sample_weight=sample_weight)),
        "PR-AUC": float(average_precision_score(y, probability, sample_weight=sample_weight)),
        "Brier score": float(brier_score_loss(y, probability, sample_weight=sample_weight)),
        "Recall at <=10% false-alarm rate": recall_at_fpr(y, probability, sample_weight),
    }
def object_bootstrap(y, probability, groups, seed):
    y = np.asarray(y, dtype=np.int8)
    probability = np.asarray(probability, dtype=float)
    codes, object_ids = pd.factorize(np.asarray(groups), sort=True)
    if len(object_ids) < 2:
        raise ValueError("Object bootstrap requires at least two objects.")
    rng = np.random.default_rng(seed)
    samples = {name: [] for name in metric_values(y, probability)}
    n_objects = len(object_ids)
    for _ in range(BOOTSTRAPS):
        draws = rng.integers(0, n_objects, size=n_objects)
        multiplicity = np.bincount(draws, minlength=n_objects)
        values = metric_values(y, probability, sample_weight=multiplicity[codes])
        for name, value in values.items():
            samples[name].append(value)
    point = metric_values(y, probability)
    return {
        name: {"estimate": point[name], "low": float(np.quantile(values, .025)),
               "high": float(np.quantile(values, .975))}
        for name, values in samples.items()
    }
def locked_threshold_at_fpr(y, probability):
    fpr, tpr, thresholds = roc_curve(y, probability, drop_intermediate=False)
    allowed = np.flatnonzero(fpr <= MAX_FALSE_ALARM_RATE)
    best_recall = tpr[allowed].max()
    best = allowed[np.flatnonzero(tpr[allowed] == best_recall)[0]]
    return float(thresholds[best])
train_data = data.loc[train].reset_index(drop=True)
valid_data = data.loc[valid].reset_index(drop=True)
y_train = train_data[TARGET].astype("int8").to_numpy()
groups_train = train_data["norad_cat_id"].to_numpy()
cv = GroupKFold(n_splits=5)
models = {
    "Logistic regression without solar": (NO_SUN, "logistic"),
    "Logistic regression with solar": (WITH_SUN, "logistic"),
    "Histogram gradient boosting without solar": (NO_SUN, "boost"),
    "Histogram gradient boosting with solar": (WITH_SUN, "boost"),
}
fitted, predictions, summaries, thresholds = {}, {}, {}, {}
for model_number, (name, (features, algorithm)) in enumerate(models.items(), start=1):
    print(f"\n{name}: grouped five-fold OOF training and full train fit")
    template = make_model(features, algorithm)
    oof = np.full(len(train_data), np.nan, dtype=float)
    for fold, (fit_idx, score_idx) in enumerate(cv.split(train_data[features], y_train, groups_train), start=1):
        fold_model = clone(template)
        with threadpool_limits(limits=4):
            fold_model.fit(train_data.iloc[fit_idx][features], y_train[fit_idx])
            oof[score_idx] = fold_model.predict_proba(train_data.iloc[score_idx][features])[:, 1]
        print(f"  completed fold {fold}/5")
    assert np.isfinite(oof).all()
    with threadpool_limits(limits=4):
        template.fit(train_data[features], y_train)
        validation_probability = template.predict_proba(valid_data[features])[:, 1]
    train_summary = object_bootstrap(y_train, oof, groups_train, SEED + model_number)
    valid_summary = object_bootstrap(
        valid_data[TARGET].astype("int8").to_numpy(), validation_probability,
        valid_data["norad_cat_id"].to_numpy(), SEED + 100 + model_number,
    )
    fitted[name] = template
    predictions[name] = {"train_oof": oof, "validation": validation_probability}
    summaries[name] = {"Training CV (OOF)": train_summary, "Validation": valid_summary}
    thresholds[name] = locked_threshold_at_fpr(valid_data[TARGET], validation_probability)
    for split_name, values in summaries[name].items():
        print(f"  {split_name}")
        for metric, record in values.items():
            print(f"    {metric}: {record['estimate']:.4f} [95% object bootstrap CI: {record['low']:.4f} to {record['high']:.4f}]")
    print(f"  validation threshold locked at FPR<=10%: {thresholds[name]:.6g}")
def baseline_physics(frame):
    perigee = pd.to_numeric(frame["perigee_km_asof"], errors="coerce").to_numpy()
    rate = pd.to_numeric(frame["perigee_decay_rate_km_per_day"], errors="coerce").to_numpy()
    days_to_120 = np.divide(perigee - 120.0, -rate, out=np.full(len(frame), np.inf), where=rate < 0)
    return ((perigee <= 120.0) | ((rate < 0) & (days_to_120 <= 365.0))).astype(float)
print("\nValidation baselines (same metrics and object bootstrap):")
baseline_rows = []
for name, probability in [
    ("Training-rate baseline", np.full(len(valid_data), y_train.mean(), dtype=float)),
    ("Perigee-decay physics baseline", baseline_physics(valid_data)),
]:
    result = object_bootstrap(valid_data[TARGET], probability, valid_data["norad_cat_id"], SEED + 500)
    baseline_rows.append({"Model": name, **{m: result[m]["estimate"] for m in result}})
    print(name)
    for metric, record in result.items():
        print(f"  {metric}: {record['estimate']:.4f} [95% object bootstrap CI: {record['low']:.4f} to {record['high']:.4f}]")
def paired_delta_bootstrap(y, first, second, groups, seed):
    y = np.asarray(y, dtype=np.int8)
    codes, object_ids = pd.factorize(np.asarray(groups), sort=True)
    rng = np.random.default_rng(seed)
    deltas = {m: [] for m in metric_values(y, first)}
    for _ in range(BOOTSTRAPS):
        multiplicity = np.bincount(rng.integers(0, len(object_ids), len(object_ids)), minlength=len(object_ids))
        weight = multiplicity[codes]
        a = metric_values(y, first, weight)
        b = metric_values(y, second, weight)
        for metric in deltas:
            deltas[metric].append(b[metric] - a[metric])
    return {m: (float(np.quantile(v, .025)), float(np.quantile(v, .975))) for m, v in deltas.items()}
print("\nSun minus no-Sun paired object-bootstrap deltas (positive means higher metric; Brier lower is better):")
sun_comparisons = {}
for algorithm_label, no_sun_name, with_sun_name in [
    ("Logistic regression", "Logistic regression without solar", "Logistic regression with solar"),
    ("Histogram gradient boosting", "Histogram gradient boosting without solar", "Histogram gradient boosting with solar"),
]:
    sun_comparisons[algorithm_label] = {}
    for split_name, key in [("Training CV (OOF)", "train_oof"), ("Validation", "validation")]:
        y = y_train if key == "train_oof" else valid_data[TARGET].to_numpy()
        groups = groups_train if key == "train_oof" else valid_data["norad_cat_id"].to_numpy()
        delta = paired_delta_bootstrap(
            y, predictions[no_sun_name][key], predictions[with_sun_name][key], groups,
            SEED + (600 if key == "train_oof" else 700),
        )
        sun_comparisons[algorithm_label][split_name] = delta
        point_a = metric_values(y, predictions[no_sun_name][key])
        point_b = metric_values(y, predictions[with_sun_name][key])
        print(f"{algorithm_label} {split_name}")
        for metric, ci in delta.items():
            print(f"  {metric} change (with Sun minus without Sun): {point_b[metric]-point_a[metric]:+.4f} [95% paired object bootstrap CI: {ci[0]:+.4f} to {ci[1]:+.4f}]")
print("\nF10.7 observed trailing mean coverage and range (train-derived activity bands):")
f107 = pd.to_numeric(train_data["f107_adj_mean_81d"], errors="coerce")
cut1, cut2 = f107.quantile([1/3, 2/3]).to_numpy()
solar_bins = [-np.inf, cut1, cut2, np.inf]
solar_labels = ["low", "middle", "high"]
solar_band_counts = {}
for split_name, frame in [("Train", train_data), ("Validation", valid_data)]:
    values = pd.to_numeric(frame["f107_adj_mean_81d"], errors="coerce")
    bands = pd.cut(values, bins=solar_bins, labels=solar_labels, include_lowest=True)
    counts = bands.value_counts().reindex(solar_labels, fill_value=0)
    objects_by_band = frame.assign(_band=bands).groupby("_band", observed=False)["norad_cat_id"].nunique()
    solar_band_counts[split_name] = {label: int(objects_by_band.get(label, 0)) for label in solar_labels}
    print(f"{split_name}: F10.7 n={values.notna().sum():,}; min={values.min():.2f}; "
          f"median={values.median():.2f}; max={values.max():.2f}; rows by band={counts.to_dict()}; "
          f"objects by band={solar_band_counts[split_name]}")
valid_bands = sum(count > 0 for count in solar_band_counts["Validation"].values())
if valid_bands <= 1:
    print("Sun question is untested across solar activity bands: validation covers one band or fewer.")
else:
    print(f"Validation spans {valid_bands} training-defined solar activity bands.")
validation_ap = {name: summaries[name]["Validation"]["PR-AUC"]["estimate"] for name in models}
validation_brier = {name: summaries[name]["Validation"]["Brier score"]["estimate"] for name in models}
selected_model_name = sorted(models, key=lambda name: (-validation_ap[name], validation_brier[name]))[0]
print(f"\nSelected by validation rule (highest PR-AUC, then lowest Brier): {selected_model_name}")
artifact_dir = DATA_DIR / "model_artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_manifest = {
    "target": TARGET,
    "scikit_learn_version": sklearn.__version__,
    "seed": SEED,
    "decision_rule": "select highest validation PR-AUC, tie-break on lower validation Brier",
    "selected_model": selected_model_name,
    "fpr_limit": MAX_FALSE_ALARM_RATE,
    "solar_band_cutpoints_f107_adj_mean_81d": [float(cut1), float(cut2)],
    "training_rate": float(y_train.mean()),
    "models": {},
}
for name, (features, algorithm) in models.items():
    slug = name.lower().replace(" ", "_")
    path = artifact_dir / f"{slug}.joblib"
    joblib.dump(fitted[name], path)
    artifact_manifest["models"][name] = {
        "artifact": path.name,
        "algorithm": algorithm,
        "features": features,
        "validation_threshold_at_10pct_fpr": thresholds[name],
        "validation_metrics": {m: v["estimate"] for m, v in summaries[name]["Validation"].items()},
    }
(artifact_dir / "manifest.json").write_text(json.dumps(artifact_manifest, indent=2), encoding="utf-8")
print("Saved four frozen pipelines and validation manifest under data/processed_plan/model_artifacts.")


Loaded train and validation rows only; test rows were not materialized.
Train: 122,859 rows; 3,202 objects; positive rate 0.0519
Validation: 4,954 rows; 612 objects; positive rate 0.2990

Logistic regression without solar: grouped five-fold OOF training and full train fit
  completed fold 1/5
  completed fold 2/5
  completed fold 3/5
  completed fold 4/5
  completed fold 5/5
  Training CV (OOF)
    Accuracy (threshold 0.50): 0.9137 [95% object bootstrap CI: 0.9044 to 0.9236]
    Precision (threshold 0.50): 0.3660 [95% object bootstrap CI: 0.3385 to 0.3951]
    Recall (threshold 0.50): 0.9045 [95% object bootstrap CI: 0.8900 to 0.9185]
    ROC AUC: 0.9663 [95% object bootstrap CI: 0.9613 to 0.9708]
    PR-AUC: 0.7443 [95% object bootstrap CI: 0.7212 to 0.7645]
    Brier score: 0.0660 [95% object bootstrap CI: 0.0606 to 0.0709]
    Recall at <=10% false-alarm rate: 0.9200 [95% object bootstrap CI: 0.9032 to 0.9363]
  Validation
    Accuracy (threshold 0.50): 0.8777 [95% object bootstrap 

## Interpretation and handoff

The selected model and per-model 10% false-alarm thresholds are locked using validation. The B2 panel and generated model artifacts remain local and must not be committed. Run the test notebook only after these choices are frozen. Grouped bootstrap intervals resample objects, keeping each object's monthly observations together.

## Validation error analysis and feature diagnostics

These diagnostics use validation only; no test records are loaded here. Logistic permutation results measure predictive contribution under shuffling, not causality. The model summary reports every PLAN classification metric for every fitted candidate.


In [ ]:
import json
from pathlib import Path
import duckdb
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/processed_plan/panel.csv").is_file())
DATA_DIR = ROOT / "data/processed_plan"
ARTIFACT_DIR = DATA_DIR / "model_artifacts"
manifest = json.loads((ARTIFACT_DIR / "manifest.json").read_text(encoding="utf-8"))
feature_groups = json.loads((DATA_DIR / "feature_groups.json").read_text(encoding="utf-8"))
needed = set(["split", "norad_cat_id", "obs_date", "reenters_within_horizon", "f107_adj_mean_81d",
              "perigee_km_asof", "object_type", "rcs_size"])
for spec in manifest["models"].values():
    needed.update(spec["features"])
quoted = ", ".join('"' + c.replace('"', '""') + '"' for c in sorted(needed))
data = duckdb.connect().execute(
    f"SELECT {quoted} FROM read_csv_auto(?) WHERE split IN ('train', 'valid')",
    [str(DATA_DIR / "panel.csv")],
).df()
valid = data[data.split.eq("valid")].reset_index(drop=True)
target = "reenters_within_horizon"
y = valid[target].astype("int8").to_numpy()
selected_name = manifest["selected_model"]
spec = manifest["models"][selected_name]
selected_model = joblib.load(ARTIFACT_DIR / spec["artifact"])
selected_probability = selected_model.predict_proba(valid[spec["features"]])[:, 1]
selected_prediction = selected_probability >= spec["validation_threshold_at_10pct_fpr"]
print(f"\nValidation-only post-selection diagnostic for {selected_name}; threshold locked on validation at FPR<=10%.")
errors = valid[["norad_cat_id", target, "perigee_km_asof", "object_type", "rcs_size"]].copy()
errors["prediction"] = selected_prediction.astype("int8")
errors["error"] = errors.prediction.ne(errors[target].astype("int8"))
errors["error_type"] = np.select(
    [(errors.prediction == 1) & (errors[target] == 0), (errors.prediction == 0) & (errors[target] == 1)],
    ["false_positive", "false_negative"], default="correct")
errors["perigee_band"] = pd.cut(errors.perigee_km_asof,
    bins=[-np.inf, 200, 400, 600], labels=["<=200 km", "200-400 km", "400-600 km"])
cut1, cut2 = manifest["solar_band_cutpoints_f107_adj_mean_81d"]
errors["solar_band"] = pd.cut(valid["f107_adj_mean_81d"], [-np.inf, cut1, cut2, np.inf],
    labels=["low", "middle", "high"], include_lowest=True)
for group in ["perigee_band", "object_type", "rcs_size", "solar_band"]:
    table = errors.groupby(group, observed=True, dropna=False).agg(
        rows=(target, "size"), objects=("norad_cat_id", "nunique"),
        positive_rate=(target, "mean"), error_rate=("error", "mean"),
        false_positives=("error_type", lambda v: int((v == "false_positive").sum())),
        false_negatives=("error_type", lambda v: int((v == "false_negative").sum())),
    ).reset_index()
    print(f"\nValidation error analysis by {group}:")
    print(table.to_string(index=False, float_format=lambda value: f"{value:.3f}"))
print("\\nValidation model summary (input features and validation performance):")
for name, model_spec in manifest["models"].items():
    print(f"\\n{name} ({model_spec['algorithm']}) - {len(model_spec['features'])} raw input features:")
    print("  Features: " + ", ".join(model_spec["features"]))
summary = []
for name, model_spec in manifest["models"].items():
    values = model_spec["validation_metrics"]
    summary.append({
        "Model": name,
        "Algorithm": model_spec["algorithm"],
        "Features": len(model_spec["features"]),
        "Accuracy (0.50)": values["Accuracy (threshold 0.50)"],
        "Precision (0.50)": values["Precision (threshold 0.50)"],
        "Recall (0.50)": values["Recall (threshold 0.50)"],
        "ROC AUC": values["ROC AUC"],
        "PR-AUC": values["PR-AUC"],
        "Brier score": values["Brier score"],
        "Recall at <=10% FA": values["Recall at <=10% false-alarm rate"],
        "Locked threshold": model_spec["validation_threshold_at_10pct_fpr"],
    })
print("\n=== Validation model summary: all metrics ===")
print(pd.DataFrame(summary).to_string(index=False, float_format=lambda value: f"{value:.4f}"))
def logistic_permutation_report(model_name, seed):
    model_spec = manifest["models"][model_name]
    if model_spec["algorithm"] != "logistic":
        return
    model = joblib.load(ARTIFACT_DIR / model_spec["artifact"])
    columns = model_spec["features"]
    X = valid[columns]
    baseline = model.predict_proba(X)[:, 1]
    base_ap = average_precision_score(y, baseline)
    base_brier = brier_score_loss(y, baseline)
    groups = {
        "Object characteristics": feature_groups["features"]["orbit_static"],
        "Object age": feature_groups["features"]["object_age"],
        "As-of orbit": feature_groups["features"]["orbit_asof"],
        "Solar and geomagnetic weather": feature_groups["features"]["space_weather_trailing"],
    }
    groups = {name: [c for c in values if c in columns] for name, values in groups.items()}
    rng = np.random.default_rng(seed)
    rows = []
    for group_name, selected in groups.items():
        if not selected:
            continue
        ap_drops, brier_increases = [], []
        for _ in range(5):
            order = rng.permutation(len(X))
            shuffled = X.copy()
            shuffled.loc[:, selected] = X.iloc[order][selected].to_numpy()
            probability = model.predict_proba(shuffled)[:, 1]
            ap_drops.append(base_ap - average_precision_score(y, probability))
            brier_increases.append(brier_score_loss(y, probability) - base_brier)
        rows.append({"Model": model_name, "Feature group": group_name, "Inputs": len(selected),
                     "AP drop": np.mean(ap_drops), "Brier increase": np.mean(brier_increases)})
    print(f"\nGrouped validation permutation importance: {model_name}")
    print(pd.DataFrame(rows).to_string(index=False, float_format=lambda value: f"{value:.4f}"))
    individual = []
    for feature in columns:
        drops, rises = [], []
        for _ in range(3):
            order = rng.permutation(len(X))
            shuffled = X.copy()
            shuffled[feature] = X.iloc[order][feature].to_numpy()
            probability = model.predict_proba(shuffled)[:, 1]
            drops.append(base_ap - average_precision_score(y, probability))
            rises.append(brier_score_loss(y, probability) - base_brier)
        individual.append({"Feature": feature, "AP drop": np.mean(drops), "Brier increase": np.mean(rises)})
    individual = pd.DataFrame(individual).sort_values("AP drop", ascending=False).head(15)
    print("Top individual validation inputs:")
    print(individual.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
for i, name in enumerate(manifest["models"]):
    logistic_permutation_report(name, 20261116 + i)



Validation-only post-selection diagnostic for Histogram gradient boosting with solar; threshold locked on validation at FPR<=10%.

Validation error analysis by perigee_band:
perigee_band  rows  objects  positive_rate  error_rate  false_positives  false_negatives
    <=200 km    73       14          0.452       0.260               19                0
  200-400 km  1293      274          0.326       0.053               69                0
  400-600 km  3588      460          0.286       0.098              253               98

Validation error analysis by object_type:
object_type  rows  objects  positive_rate  error_rate  false_positives  false_negatives
     DEBRIS  3691      491          0.385       0.103              283               96
ROCKET BODY   937       83          0.057       0.058               53                1
    UNKNOWN   326       38          0.018       0.018                5                1

Validation error analysis by rcs_size:
rcs_size  rows  objects  positive_